# Elastic Net
- regression technique that combines Ridge and Lasso
- Ridge combats multicollinearity - my features are likely connected
- Lasso combats noise - not sure if all the features are relevant
- trying this because I think the data is linear and to see if it does better than linear regression (baseline)

In [29]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
import joblib #to save the model to be able to import it into visualization notebook
import pandas as pd

#import from the data_splitter Python script to load and split the data - prevents having to copy and paste code
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..'))) #used to move to the parent folder
from src.data_splitter import load_data, split

#load dataset and variables
df, X, y = load_data()

#split data by groups
X_train, X_val, X_test, y_train, y_val, y_test = split(df, X, y)

Training: 10661 rows | 70 unique players
Validation: 2169 rows | 15 unique players
Testing: 2170 rows | 15 unique players


In [2]:
#one hot coding for string
categorical = ["movement_type"]
numeric = [col for col in X_train.columns if col not in categorical]

#preprocess data: needs to be scaled like linear regression
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical)
    ]
)

#combine preprocessor and elastic net in a pipeline
'''
ElasticNetCV dynamically finds the best hyperparameters automatically
- alpha - regularization strength, how hard model tries to keep coefficients small to prevent overfitting
- l1_ratio - mixing parameter (of Ridge and Lasso)
    0 - pure Ridge
    1 - pure Lasso
'''
elastic_net = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", ElasticNetCV())
    ]
)

elastic_net.fit(X_train, y_train)
print("Elastic net pipeline fitted successfully on training data")

Elastic net pipeline fitted successfully on training data


In [8]:
#to know what hyperparameters it used
best_alpha = elastic_net.named_steps['regressor'].alpha_

#evaluation on elastic net regressor using validation data
#predict target values for unseen validation data
y_val_pred = elastic_net.predict(X_val)

#calculate regression metrics: mean absolute error, root mean squared error, R^2
val_mae = mean_absolute_error(y_val, y_val_pred)
val_rmse = root_mean_squared_error(y_val, y_val_pred)
val_r2 = r2_score(y_val, y_val_pred)

#performance summary
print("=== ELASTIC NET (Validation Set) ===")
print(f"Used alpha={best_alpha:.4f}")
print(f"MAE  : {val_mae:.4f}")
print(f"RMSE : {val_rmse:.4f}")
print(f"R²   : {val_r2:.4f}")

=== ELASTIC NET (Validation Set) ===
Used alpha=0.0248
MAE  : 3.1951
RMSE : 4.0290
R²   : 0.6658


In [25]:
#trying different values for l1_ratio to see if that causes any change
ratios = [0.01, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]

#loop through the different values to see which one is best
for r in ratios:
    en_pipeline_l1_ratio = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", ElasticNetCV(
                l1_ratio=r
            ))
        ]
    )
    
    #train and compare the l1_ratio
    en_pipeline_l1_ratio.fit(X_train, y_train)
    pred = en_pipeline_l1_ratio.predict(X_val)
    mae = mean_absolute_error(y_val, pred)
    rmse = root_mean_squared_error(y_val, pred)
    r2 = r2_score(y_val, pred)

    #see which alpha value it chose for each iteration
    best_alpha = en_pipeline_l1_ratio.named_steps['regressor'].alpha_
    
    print(f"alpha:{best_alpha:.4f} | l1_ratio: {str(r):<5}| MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f}")

alpha:0.3067 | l1_ratio: 0.01 | MAE: 3.3711 | RMSE: 4.2483 | R²: 0.6285
alpha:0.0307 | l1_ratio: 0.1  | MAE: 3.1998 | RMSE: 4.0347 | R²: 0.6649
alpha:0.0192 | l1_ratio: 0.3  | MAE: 3.1967 | RMSE: 4.0311 | R²: 0.6655
alpha:0.0248 | l1_ratio: 0.5  | MAE: 3.1951 | RMSE: 4.0290 | R²: 0.6658
alpha:0.0355 | l1_ratio: 0.7  | MAE: 3.1936 | RMSE: 4.0271 | R²: 0.6662
alpha:0.0483 | l1_ratio: 0.9  | MAE: 3.1924 | RMSE: 4.0261 | R²: 0.6663
alpha:0.0536 | l1_ratio: 1.0  | MAE: 3.1922 | RMSE: 4.0260 | R²: 0.6663


In [38]:
#l1_ratio=1.0 performed the best, meaning this is pure Lasso Regularization
#this indicates that there is noise in the model

en_final = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", ElasticNetCV(
            l1_ratio=1.0
        ))
    ]
)

en_final.fit(X_train, y_train)
joblib.dump(en_final, 'elastic_net.joblib')
#evaluation on elastic net regressor using validation data
#predict target values for unseen validation data
final_y_val_pred = en_final.predict(X_val)

#calculate regression metrics: mean absolute error, root mean squared error, R^2
final_val_mae = mean_absolute_error(y_val, final_y_val_pred)
final_val_rmse = root_mean_squared_error(y_val, final_y_val_pred)
final_val_r2 = r2_score(y_val, final_y_val_pred)

#to know what hyperparameters it used
best_alpha = en_final.named_steps['regressor'].alpha_

#determining how many coefficients it used
coefs = en_final.named_steps['regressor'].coef_

kept = np.sum(coefs != 0)
removed = np.sum(coefs == 0)

#performance summary
print("=== ELASTIC NET (Validation Set) ===")
print(f"Used alpha={best_alpha:.4f}, attributes kept={kept}, attributes removed={removed}")
print(f"MAE  : {final_val_mae:.4f}")
print(f"RMSE : {final_val_rmse:.4f}")
print(f"R²   : {final_val_r2:.4f}")

#list of features that were kept
coef_df["feature"] = coef_df["feature"].str.replace("num__", "", regex=False)
kept_df = coef_df[coef_df["coef"]!= 0].sort_values(by="coef", key=abs, ascending=False)
print("\n=== KEPT FEATURES (Sorted by Impact) ===")
print(kept_df.to_string(index=False))

=== ELASTIC NET (Validation Set) ===
Used alpha=0.0536, attributes kept=5, attributes removed=23
MAE  : 3.1922
RMSE : 4.0260
R²   : 0.6663

=== KEPT FEATURES (Sorted by Impact) ===
                    feature     coef
   movement_stability_score 2.961151
              balance_score 2.940895
    vertical_jump_height_cm 2.158467
center_of_mass_velocity_m_s 2.137189
      movement_symmetry_pct 1.958523


# Conclusion
- tried ElasticNet to see if it could break baseline linear regression
- extremely low alpha (close to 0), indicating baseline wasn't really suffering from overfitting but needed to focus on less input features
- l1_ratio=1.0 so it's Lasso Regularization
- performs better than baseline Linear Regression, indicating that there is noise (could be redundant features) in the model
- only 5 features were kept, indicating a lot of irrelevant/redundant features that could be safely pruned
- able to do better than baseline linear regression